## Section 1 — Préparation des données

### Chargement des librairies nécessaires

In [1]:
import pandas as pd
import spacy
import re

### Chargement des datas

In [2]:
avis = [
 ("Super plateforme , j’ai beaucoup appris !!! :)", 5),
 ("TRES DECEVANT ... le support ne repond jamais.", 1),
 ("Bien mais un peu cher pour ce que c’est.", 3),
 ("<p>Interface claire et agreable a utiliser.</p>", 4),
 ("nul , ca plante tout le temps http :// formaplus.example.com/bug", 1),
 ("Je recommande vivement , contenu de qualite :)", 5),
 ("Pas mal du tout , quelques bugs mineurs.", 4),
 ("Service client injoignable , tres mecontent !!!", 1),
 ("Excellent rapport qualite -prix , a essayer.", 5),
 ("Bien mais un peu cher pour ce que c’est.", 3),
 ("Les videos sont trop courtes a mon gout.", 3),
 ("Je ne suis pas satisfait du tout , remboursez -moi.", 1),
 ("Application fluide , aucun souci depuis 3 mois.", 5),
 ("Moyen ... l’ergonomie pourrait etre amelioree.", 2),
 ("PARFAIT ! exactement ce qu’il me fallait <br> Merci", 5),
 ("Corect sans plus , rien d’exceptionnel", 3),
 ("Trop de publicites , ca devient agacant.", 2),
 ("Une vraie pepite pour apprendre le NLP", 5),
 ]

### Importation, chargement et exploration du jeux de données

In [3]:
# Transformation des avis en dataframe
df_init = pd.DataFrame(avis , columns =["avis", "note"])

# Sauvegarde du dataframe en fichier csv
df_init.to_csv("corpus_avis.csv", index=False , encoding="utf -8")
print(f"Shape of the DataFrame: {df_init.shape}")

Shape of the DataFrame: (18, 2)


In [4]:
# Chargement du csv
df = pd.read_csv("corpus_avis.csv", index_col=False, encoding="utf -8")

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   avis    18 non-null     str  
 1   note    18 non-null     int64
dtypes: int64(1), str(1)
memory usage: 420.0 bytes


In [6]:
# Affichage du DataFrame
print(df)

                                                 avis  note
0      Super plateforme , j’ai beaucoup appris !!! :)     5
1      TRES DECEVANT ... le support ne repond jamais.     1
2            Bien mais un peu cher pour ce que c’est.     3
3     <p>Interface claire et agreable a utiliser.</p>     4
4   nul , ca plante tout le temps http :// formapl...     1
5      Je recommande vivement , contenu de qualite :)     5
6            Pas mal du tout , quelques bugs mineurs.     4
7     Service client injoignable , tres mecontent !!!     1
8        Excellent rapport qualite -prix , a essayer.     5
9            Bien mais un peu cher pour ce que c’est.     3
10           Les videos sont trop courtes a mon gout.     3
11  Je ne suis pas satisfait du tout , remboursez ...     1
12    Application fluide , aucun souci depuis 3 mois.     5
13     Moyen ... l’ergonomie pourrait etre amelioree.     2
14  PARFAIT ! exactement ce qu’il me fallait <br> ...     5
15             Corect sans plus , rien d

In [7]:
# Affichage des métriques
df.describe()

,note
count,18.000000
mean,3.222222
std,1.592466
min,1.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,5.000000


### Nettoyage du jeux de données

In [8]:
def clean_text(text):
    """
    Cette fonction nettoie le texte en supprimant les balises HTML, les URL, la ponctuation excessive et les emojis. 
    Elle convertit également le texte en minuscules et supprime les espaces multiples.
    """
# Convertir en minuscules
    text = text.lower()
    
    # Suppression des balises HTML
    text = re.sub(r"<.*?>", " ", text)
    
    # Suppression des URL
    text = re.sub(r"https?\s*:\s*/\s*/\s*\S+", " ", text)
    
    # Suppression de la ponctuation excessive
    text = re.sub(r"[!]{2,}", "!", text)
    text = re.sub(r"[.]{2,}", ".", text)
    
    # Suppression des espaces multiples et des espaces en début ou fin
    text = re.sub(r" +", " ", text)
    text = text.strip()
    
    # Suppression des emojis (optionnel)
    text = re.sub(r"[^\w\s]", "", text)

    return text

In [9]:
df["avis_clean"] = df["avis"].apply(clean_text)
(df[["avis", "avis_clean"]].head(10))

,avis,avis_clean
0,"Super plateforme , j’ai beaucoup appris !!! :)",super plateforme jai beaucoup appris
1,TRES DECEVANT ... le support ne repond jamais.,tres decevant le support ne repond jamais
2,Bien mais un peu cher pour ce que c’est.,bien mais un peu cher pour ce que cest
3,<p>Interface claire et agreable a utiliser.</p>,interface claire et agreable a utiliser
4,"nul , ca plante tout le temps http :// formapl...",nul ca plante tout le temps
5,"Je recommande vivement , contenu de qualite :)",je recommande vivement contenu de qualite
6,"Pas mal du tout , quelques bugs mineurs.",pas mal du tout quelques bugs mineurs
7,"Service client injoignable , tres mecontent !!!",service client injoignable tres mecontent
8,"Excellent rapport qualite -prix , a essayer.",excellent rapport qualite prix a essayer
9,Bien mais un peu cher pour ce que c’est.,bien mais un peu cher pour ce que cest


Après nettoyage nous obtenons la colonne avis_clean qui est débarasser des URLs, ponctuation et informations peu nécéssaire a la suite du traitement.

In [10]:
# Suppression des doublons
avant = df.shape[0]
df.drop_duplicates(subset=["avis_clean"], inplace=True)
apres = df.shape[0]
print(f"Nombre de commanentaires avant suppression des doublons : {avant}")
print(f"Nombre de commanentaires après suppression des doublons : {apres}")
print(f"Nombre de doublons supprimés : {avant - apres}")

Nombre de commanentaires avant suppression des doublons : 18
Nombre de commanentaires après suppression des doublons : 17
Nombre de doublons supprimés : 1


### Tokenisation et lemmatisation avec spaCy

Cette étape consiste à transformer une phrase en une liste de mots et à appliquer la lemmatization qui consiste à ramener un mot à sa racine.

In [11]:
# Chargement du modèle de langue française de spaCy
nlp = spacy.load("fr_core_news_sm")

In [12]:
def tokenize_lemmatize(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc if token.is_alpha]

In [13]:
df["tokens"] = df["avis_clean"].apply(tokenize_lemmatize)
df[["avis_clean", "tokens"]].head(3)

,avis_clean,tokens
0,super plateforme jai beaucoup appris,"[super, plateform, jai, beaucoup, apprendre]"
1,tres decevant le support ne repond jamais,"[tre, decever, le, support, ne, repond, jamais]"
2,bien mais un peu cher pour ce que cest,"[bien, mais, un, peu, cher, pour, ce, que, cest]"


### Suppression des mots vides (avec prudence)

Les stopwords representent l'ensemble des mots qui n'apportent de contexte en plus dans le traitement NLP c'est pourquoi il est nécéssaire de les rétirer de la liste des tokens.

In [14]:
# Suppression des stopwords (conservation de "ne", "pas", "non" et "sans")
stop_words = nlp.Defaults.stop_words - {"ne", "pas", "non", "sans"}

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

In [15]:
df["tokens_final"] = df["tokens"].apply(remove_stopwords)
df[["avis_clean", "tokens_final"]].head(3)

,avis_clean,tokens_final
0,super plateforme jai beaucoup appris,"[super, plateform, jai, beaucoup, apprendre]"
1,tres decevant le support ne repond jamais,"[tre, decever, support, ne, repond, jamais]"
2,bien mais un peu cher pour ce que cest,"[bien, cher, cest]"


### Statistiques rapides : Avant et Après suppression des stopwords

In [16]:
nb_tokens_avant = df["avis_clean"].apply(lambda t: len(t.split())).sum()
nb_tokens_apres = df["tokens_final"].apply(len).sum()
vocabulaire = set(t for tokens in df["tokens_final"] for t in tokens)

print(f"Tokens avant nettoyage des mots vides : {nb_tokens_avant}")
print(f"Tokens apres nettoyage des mots vides : {nb_tokens_apres}")
print(f"Taille du vocabulaire final : {len(vocabulaire)}")

Tokens avant nettoyage des mots vides : 111
Tokens apres nettoyage des mots vides : 72
Taille du vocabulaire final : 66


Après le processus dee traitement, nous constatons qu'une trentaine de mots n'était pas nécessaire pour la suite de notre analyse (nous passons de 111 mots à 72 mots), et notre vocabulaire final est de 66 mots.

In [17]:
print(f"Vocabulaire final : {vocabulaire}")

Vocabulaire final : {'rien', 'jai', 'support', 'qualit', 'mois', 'parfaire', 'application', 'dexceptionnel', 'recommander', 'devient', 'injoignable', 'amelioree', 'corect', 'souci', 'tre', 'clair', 'jamais', 'falloir', 'pepit', 'gout', 'nlp', 'contenu', 'agreabl', 'cher', 'beaucoup', 'pas', 'decever', 'fluide', 'aucun', 'prix', 'ca', 'satisfait', 'vraie', 'ne', 'quil', 'interface', 'bug', 'repond', 'temps', 'rembourser', 'plateform', 'qualite', 'sans', 'utiliser', 'super', 'apprendre', 'plante', 'publicite', 'pouvoir', 'excellent', 'client', 'bien', 'court', 'service', 'mal', 'video', 'trop', 'meconter', 'agacer', 'rapport', 'lergonomie', 'mineur', 'cest', 'moyen', 'essayer', 'vivement'}


**Resultats NLP Pipeline**
* Tokens avant nettoyage des mots vides :   **113**
* Tokens apres nettoyage des mots vides :   **74**
* Taille du vocabulaire final :             **66**

## Documentation du TP 1
---

### Source du corpus
Le jeux de données utilisé est un ensemble de commentaires semblant provenir d'une application de formation en ligne.

### Volume
Avant nettoyage des doublons, le jeux de données comprenait **18 commentaires**.
Après suppression, il en contenait 17, soit **1 doublon supprimé**.

### Bruit identifié
Le jeux de données initial contenait plusieurs anomalies : 
- Plusieurs espaces au milieu des phrases
- Répétition exagéré des ponctuations
- Des balises HTML

### Règles de nettoyages appliqués
Le nettoyage appliqué suit plusieurs étapes :
- Suppression des doublons 
- Transformation du texte en miniscule pour uniformiser l'ensemble
- Suppression des balises
- Suppression des liens
- Suppression des émojis
- Suppression de la ponctuation excessive
- Suppression des espaces multiples

### Choix de la normalisation
Pour le traitement de ces commentaires en langue française, nous avons fait le choix de conserver les accents car ces derniers apportent au texte un sens pour la suite de l'analyse.

La casse et la ponctuation seront les éléments que nous normaliseront de notre étude avec la méthode de lemmatization.


### Traitement des stopwords et de la négation
La négation sera conserver pour ne pas perdre le contexte de certains commentaires.

Quant au stopwords, ils ne seront pas conserver car ils n'apportent pas du contexte en plus dans l'étude de ces commentaires.

### Taille du vocabulaire
Le vocabulaire final est de 66 mots

## Bonus : Stemming aved NLTK

In [18]:
# Chargement du stemmer français de NLTK
import nltk
from nltk.stem.snowball import SnowballStemmer

In [19]:
# Telechargement du corpus français pour le stemmer si nécessaire
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rashops\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
# Chargement du corpus français
from nltk.corpus import stopwords

In [21]:
# Création d'une instance du stemmer français
stemmer = SnowballStemmer("french")

In [22]:
def clean_text_stem(text):
    # Minuscules
    text = text.lower()

    # Suppression des balises HTML
    text = re.sub(r"<.*?>", " ", text)

    # Suppression des URL
    text = re.sub(r"https?\s*:\s*/\s*/\s*\S+", " ", text)

    # Suppression des espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    # Suppression des emojis (optionnel)
    text = re.sub(r"[^\w\s]", "", text)
    
    # Tokenisation simple
    words = text.split()
        
    # Stemming
    words = [stemmer.stem(word) for word in words]

    return " ".join(words)


In [23]:
df["stemming_avis_clean"] = df["avis"].apply(clean_text_stem)
df[["avis", "stemming_avis_clean"]].head(10)

,avis,stemming_avis_clean
0,"Super plateforme , j’ai beaucoup appris !!! :)",sup plateform jai beaucoup appris
1,TRES DECEVANT ... le support ne repond jamais.,tre decev le support ne repond jam
2,Bien mais un peu cher pour ce que c’est.,bien mais un peu cher pour ce que cest
3,<p>Interface claire et agreable a utiliser.</p>,interfac clair et agreabl a utilis
4,"nul , ca plante tout le temps http :// formapl...",nul ca plant tout le temp
5,"Je recommande vivement , contenu de qualite :)",je recommand viv contenu de qualit
6,"Pas mal du tout , quelques bugs mineurs.",pas mal du tout quelqu bug mineur
7,"Service client injoignable , tres mecontent !!!",servic client injoign tre mecontent
8,"Excellent rapport qualite -prix , a essayer.",excellent rapport qualit prix a essai
10,Les videos sont trop courtes a mon gout.,le videos sont trop court a mon gout


In [24]:
# Fonction simple de tokenisation
def tokenize(text):
    return [token for token in text.split()]

In [25]:
df["stem_tokens"] = df["stemming_avis_clean"].apply(tokenize)
df[["stemming_avis_clean", "stem_tokens"]].head(3)

,stemming_avis_clean,stem_tokens
0,sup plateform jai beaucoup appris,"[sup, plateform, jai, beaucoup, appris]"
1,tre decev le support ne repond jam,"[tre, decev, le, support, ne, repond, jam]"
2,bien mais un peu cher pour ce que cest,"[bien, mais, un, peu, cher, pour, ce, que, cest]"


In [26]:
stop_words = set(stopwords.words("french")) - {"ne", "pas", "non", "sans"}

def stem_remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

In [27]:
df["stem_tokens_final"] = df["stem_tokens"].apply(stem_remove_stopwords)
df[["stemming_avis_clean", "stem_tokens_final"]].head(3)

,stemming_avis_clean,stem_tokens_final
0,sup plateform jai beaucoup appris,"[sup, plateform, jai, beaucoup, appris]"
1,tre decev le support ne repond jam,"[tre, decev, support, ne, repond, jam]"
2,bien mais un peu cher pour ce que cest,"[bien, peu, cher, cest]"


In [28]:
nb_tokens_avant = df["avis_clean"].apply(lambda t: len(t.split())).sum()
nb_tokens_apres = df["stem_tokens_final"].apply(len).sum()
vocabulaire = set(t for tokens in df["stem_tokens_final"] for t in tokens)
    
print(f"Tokens avant nettoyage des mots vides : {nb_tokens_avant}")
print(f"Tokens apres nettoyage des mots vides : {nb_tokens_apres}")
print(f"Taille du vocabulaire final : {len(vocabulaire)}")

Tokens avant nettoyage des mots vides : 111
Tokens apres nettoyage des mots vides : 87
Taille du vocabulaire final : 77


In [29]:
print(f"Vocabulaire final : {vocabulaire}")

Vocabulaire final : {'depuis', 'clair', 'nlp', 'pas', 'agreabl', 'mecontent', '3', 'bug', 'bien', 'peu', 'mal', 'nul', 'plus', 'apprendr', 'essai', 'injoign', 'cest', 'moyen', 'dexceptionnel', 'recommand', 'ameliore', 'beaucoup', 'aucun', 'a', 'interfac', 'pourr', 'jam', 'client', 'temp', 'trop', 'rapport', 'utilis', 'decev', 'viv', 'qualit', 'support', 'rembours', 'corect', 'tre', 'appris', 'parf', 'prix', 'ca', 'souc', 'repond', 'quelqu', 'publicit', 'etre', 'appliqu', 'sup', 'vrai', 'merc', 'fall', 'mineur', 'satisf', 'agac', 'rien', 'jai', 'mois', 'lergonom', 'pepit', 'gout', 'contenu', 'cher', 'videos', 'ne', 'plant', 'quil', 'plateform', 'servic', 'exact', 'excellent', 'court', 'fluid', 'tout', 'san', 'devient'}


### Conclusion d'étude avec le stemming
Dans ce cas précis le stemming rend les mots et le contexte incompréhensible, donc n'est pas adapté pour notre étude.

In [30]:
df.head()

,avis,note,avis_clean,tokens,tokens_final,stemming_avis_clean,stem_tokens,stem_tokens_final
0,"Super plateforme , j’ai beaucoup appris !!! :)",5,super plateforme jai beaucoup appris,"[super, plateform, jai, beaucoup, apprendre]","[super, plateform, jai, beaucoup, apprendre]",sup plateform jai beaucoup appris,"[sup, plateform, jai, beaucoup, appris]","[sup, plateform, jai, beaucoup, appris]"
1,TRES DECEVANT ... le support ne repond jamais.,1,tres decevant le support ne repond jamais,"[tre, decever, le, support, ne, repond, jamais]","[tre, decever, support, ne, repond, jamais]",tre decev le support ne repond jam,"[tre, decev, le, support, ne, repond, jam]","[tre, decev, support, ne, repond, jam]"
2,Bien mais un peu cher pour ce que c’est.,3,bien mais un peu cher pour ce que cest,"[bien, mais, un, peu, cher, pour, ce, que, cest]","[bien, cher, cest]",bien mais un peu cher pour ce que cest,"[bien, mais, un, peu, cher, pour, ce, que, cest]","[bien, peu, cher, cest]"
3,<p>Interface claire et agreable a utiliser.</p>,4,interface claire et agreable a utiliser,"[interface, clair, et, agreabl, avoir, utiliser]","[interface, clair, agreabl, utiliser]",interfac clair et agreabl a utilis,"[interfac, clair, et, agreabl, a, utilis]","[interfac, clair, agreabl, a, utilis]"
4,"nul , ca plante tout le temps http :// formapl...",1,nul ca plante tout le temps,"[nul, ca, plante, tout, le, temps]","[ca, plante, temps]",nul ca plant tout le temp,"[nul, ca, plant, tout, le, temp]","[nul, ca, plant, tout, temp]"
